# Hospital Operations, Cost & Patient Flow Intelligence

## 02 — Data Preparation and Modelling

### Objective

Prepare the raw hospital datasets for analysis by applying validated transformations, constructing fact and dimension tables and establishing a reliable analytical data model.

### Workflow

1. Load the source datasets.
2. Prepare the encounter fact table.
3. Prepare the procedure fact table.
4. Construct dimension tables.
5. Validate primary and foreign key relationships.
6. Validate date coverage and record counts.
7. Export the analytical model to `data/processed/`.

### Data modelling approach

The project uses a dimensional model with two fact tables:

- `FactEncounters` — one row per encounter.
- `FactProcedures` — one row per procedure.

Five supporting dimensions provide patient, payer, organisation, encounter-class and date information.

In [1]:
# Import libraries
import pandas as pd
import numpy as np
from pathlib import Path

# Define project directories
PROJECT_DIR = Path.cwd().parent
RAW_DATA_DIR = PROJECT_DIR / "data" / "raw"
PROCESSED_DATA_DIR = PROJECT_DIR / "data" / "processed"

# Create processed directory if necessary
PROCESSED_DATA_DIR.mkdir(parents=True, exist_ok=True)

print("Project directory:", PROJECT_DIR)
print("Raw data directory:", RAW_DATA_DIR)
print("Processed data directory:", PROCESSED_DATA_DIR)

Project directory: c:\Users\Nadia Parker\Hospital_Operations_Cost_Patient_Flow
Raw data directory: c:\Users\Nadia Parker\Hospital_Operations_Cost_Patient_Flow\data\raw
Processed data directory: c:\Users\Nadia Parker\Hospital_Operations_Cost_Patient_Flow\data\processed


In [2]:
# Load the original source datasets

datasets = {
    "encounters": pd.read_csv(RAW_DATA_DIR / "encounters.csv"),
    "patients": pd.read_csv(RAW_DATA_DIR / "patients.csv"),
    "procedures": pd.read_csv(RAW_DATA_DIR / "procedures.csv"),
    "payers": pd.read_csv(RAW_DATA_DIR / "payers.csv"),
    "organizations": pd.read_csv(RAW_DATA_DIR / "organizations.csv")
}

# Create independent working copies
encounters = datasets["encounters"].copy()
procedures = datasets["procedures"].copy()
patients = datasets["patients"].copy()
payers = datasets["payers"].copy()
organizations = datasets["organizations"].copy()

# Confirm successful loading
for name, dataframe in datasets.items():
    print(f"{name}: {dataframe.shape}")

encounters: (27891, 14)
patients: (974, 20)
procedures: (47701, 9)
payers: (10, 7)
organizations: (1, 8)


In [3]:
# Check that all seven previously exported model tables exist

required_files = [
    "fact_encounters.csv",
    "fact_procedures.csv",
    "dim_patients.csv",
    "dim_payers.csv",
    "dim_organizations.csv",
    "dim_encounter_class.csv",
    "dim_date.csv"
]

for filename in required_files:
    file_path = PROCESSED_DATA_DIR / filename
    print(filename, "|", "Found" if file_path.exists() else "Missing")

fact_encounters.csv | Found
fact_procedures.csv | Found
dim_patients.csv | Found
dim_payers.csv | Found
dim_organizations.csv | Found
dim_encounter_class.csv | Found
dim_date.csv | Found


## 1. Prepare the Encounter Fact Table

The encounter fact table is designed at the grain of one row per healthcare encounter.

It retains the encounter identifiers, foreign keys, encounter classification, timestamps and financial measures. Additional fields are derived to support duration and time-based analysis.

In [4]:
# Select relevant encounter fields
fact_encounters = encounters[
    [
        "Id",
        "PATIENT",
        "PAYER",
        "ORGANIZATION",
        "ENCOUNTERCLASS",
        "START",
        "STOP",
        "BASE_ENCOUNTER_COST",
        "TOTAL_CLAIM_COST",
        "PAYER_COVERAGE"
    ]
].copy()

# Convert timestamps to datetime
fact_encounters["START"] = pd.to_datetime(
    fact_encounters["START"], utc=True
)

fact_encounters["STOP"] = pd.to_datetime(
    fact_encounters["STOP"], utc=True
)

# Calculate encounter duration
fact_encounters["DURATION_HOURS"] = (
    fact_encounters["STOP"] - fact_encounters["START"]
).dt.total_seconds() / 3600

# Derive date fields
fact_encounters["START_DATE"] = (
    fact_encounters["START"].dt.date
)

fact_encounters["STOP_DATE"] = (
    fact_encounters["STOP"].dt.date
)

# Flag encounters exceeding 24 hours
fact_encounters["OVER_24_HOURS"] = (
    fact_encounters["DURATION_HOURS"] > 24
)

display(fact_encounters.head())

print("Encounter records:", len(fact_encounters))
print("Unique encounter IDs:", fact_encounters["Id"].nunique())
print("Missing encounter IDs:", fact_encounters["Id"].isna().sum())

,Id,PATIENT,PAYER,ORGANIZATION,ENCOUNTERCLASS,START,STOP,BASE_ENCOUNTER_COST,TOTAL_CLAIM_COST,PAYER_COVERAGE,DURATION_HOURS,START_DATE,STOP_DATE,OVER_24_HOURS
0,32c84703-2481-49cd-d571-3899d5820253,3de74169-7f67-9304-91d4-757e0f3a14d2,b1c428d6-4f07-31e0-90f0-68ffa6ff8c76,d78e84ec-30aa-3bba-a33a-f29a3a454662,ambulatory,2011-01-02 09:26:36+00:00,2011-01-02 12:58:36+00:00,85.55,1018.02,0.00,3.533333,2011-01-02,2011-01-02,False
1,c98059da-320a-c0a6-fced-c8815f3e3f39,d9ec2e44-32e9-9148-179a-1653348cc4e2,b1c428d6-4f07-31e0-90f0-68ffa6ff8c76,d78e84ec-30aa-3bba-a33a-f29a3a454662,outpatient,2011-01-03 05:44:39+00:00,2011-01-03 06:01:42+00:00,142.58,2619.36,0.00,0.284167,2011-01-03,2011-01-03,False
2,4ad28a3a-2479-782b-f29c-d5b3f41a001e,73babadf-5b2b-fee7-189e-6f41ff213e01,7caa7254-5050-3b5e-9eae-bd5ea30e809c,d78e84ec-30aa-3bba-a33a-f29a3a454662,outpatient,2011-01-03 14:32:11+00:00,2011-01-03 14:47:11+00:00,85.55,461.59,305.27,0.250000,2011-01-03,2011-01-03,False
3,c3f4da61-e4b4-21d5-587a-fbc89943bc19,3b46a0b7-0f34-9b9a-c319-ace4a1f58c0b,b1c428d6-4f07-31e0-90f0-68ffa6ff8c76,d78e84ec-30aa-3bba-a33a-f29a3a454662,wellness,2011-01-03 16:24:45+00:00,2011-01-03 16:39:45+00:00,136.80,1784.24,0.00,0.250000,2011-01-03,2011-01-03,False
4,a9183b4f-2572-72ea-54c2-b3cd038b4be7,fa006887-d93c-d302-8b89-f3c25f88c0e1,42c4fca7-f8a9-3cd1-982a-dd9751bf3e2a,d78e84ec-30aa-3bba-a33a-f29a3a454662,ambulatory,2011-01-03 17:36:53+00:00,2011-01-03 17:51:53+00:00,85.55,234.72,0.00,0.250000,2011-01-03,2011-01-03,False


Encounter records: 27891
Unique encounter IDs: 27891
Missing encounter IDs: 0


## 2. Prepare the Procedure Fact Table

The procedure fact table is designed at the grain of one row per recorded procedure.

It retains procedure-level costs and details while inheriting the relevant encounter attributes through the validated encounter reference.

This approach preserves the distinction between encounter-level and procedure-level financial measures and prevents encounter costs from being duplicated when an encounter contains multiple procedures.

In [5]:
# Select procedure-level fields
fact_procedures = procedures[
    [
        "PATIENT",
        "ENCOUNTER",
        "START",
        "STOP",
        "CODE",
        "DESCRIPTION",
        "BASE_COST"
    ]
].copy()

# Generate a unique procedure identifier
fact_procedures.insert(
    0, "PROCEDURE_ID", range(1, len(fact_procedures) + 1)
)

# Convert timestamps
fact_procedures["START"] = pd.to_datetime(
    fact_procedures["START"], utc=True
)

fact_procedures["STOP"] = pd.to_datetime(
    fact_procedures["STOP"], utc=True
)

# Calculate procedure duration
fact_procedures["DURATION_HOURS"] = (
    fact_procedures["STOP"] - fact_procedures["START"]
).dt.total_seconds() / 3600

# Derive procedure date
fact_procedures["PROCEDURE_DATE"] = (
    fact_procedures["START"].dt.date
)

# Flag zero-duration procedures
fact_procedures["ZERO_DURATION"] = (
    fact_procedures["DURATION_HOURS"] == 0
)

# Select encounter attributes
encounter_attributes = fact_encounters[
    [
        "Id",
        "PATIENT",
        "PAYER",
        "ORGANIZATION",
        "ENCOUNTERCLASS",
        "START_DATE"
    ]
].copy()

# Rename encounter fields
encounter_attributes = encounter_attributes.rename(
    columns={
        "Id": "ENCOUNTER",
        "PATIENT": "ENCOUNTER_PATIENT",
        "START_DATE": "ENCOUNTER_DATE"
    }
)

# Enrich procedures with encounter attributes
fact_procedures = fact_procedures.merge(
    encounter_attributes,
    on="ENCOUNTER",
    how="left",
    validate="many_to_one"
)

# Remove temporary comparison field
fact_procedures = fact_procedures.drop(
    columns=["ENCOUNTER_PATIENT"]
)

display(fact_procedures.head())

print("Procedure records:", len(fact_procedures))
print("Unique procedure IDs:", fact_procedures["PROCEDURE_ID"].nunique())
print("Unmatched encounter references:",
      (~fact_procedures["ENCOUNTER"].isin(fact_encounters["Id"])).sum())
print("Missing payer references:", fact_procedures["PAYER"].isna().sum())
print("Missing encounter classes:", fact_procedures["ENCOUNTERCLASS"].isna().sum())

,PROCEDURE_ID,PATIENT,ENCOUNTER,START,STOP,CODE,DESCRIPTION,BASE_COST,DURATION_HOURS,PROCEDURE_DATE,ZERO_DURATION,PAYER,ORGANIZATION,ENCOUNTERCLASS,ENCOUNTER_DATE
0,1,3de74169-7f67-9304-91d4-757e0f3a14d2,32c84703-2481-49cd-d571-3899d5820253,2011-01-02 09:26:36+00:00,2011-01-02 12:58:36+00:00,265764009,Renal dialysis (procedure),903,3.533333,2011-01-02,False,b1c428d6-4f07-31e0-90f0-68ffa6ff8c76,d78e84ec-30aa-3bba-a33a-f29a3a454662,ambulatory,2011-01-02
1,2,d9ec2e44-32e9-9148-179a-1653348cc4e2,c98059da-320a-c0a6-fced-c8815f3e3f39,2011-01-03 05:44:39+00:00,2011-01-03 06:01:42+00:00,76601001,Intramuscular injection,2477,0.284167,2011-01-03,False,b1c428d6-4f07-31e0-90f0-68ffa6ff8c76,d78e84ec-30aa-3bba-a33a-f29a3a454662,outpatient,2011-01-03
2,3,d856d6e6-4c98-e7a2-129b-44076c63d008,2cfd4ddd-ad13-fe1e-528b-15051cea2ec3,2011-01-04 14:49:55+00:00,2011-01-04 15:04:55+00:00,703423002,Combined chemotherapy and radiation therapy (p...,11620,0.250000,2011-01-04,False,7c4411ce-02f1-39b5-b9ec-dfbea9ad3c1a,d78e84ec-30aa-3bba-a33a-f29a3a454662,ambulatory,2011-01-04
3,4,bc9d59c3-0a30-6e3b-f47d-022e4f03c8de,17966936-0878-f4db-128b-a43ae10d0878,2011-01-05 04:02:09+00:00,2011-01-05 04:17:09+00:00,173160006,Diagnostic fiberoptic bronchoscopy (procedure),9796,0.250000,2011-01-05,False,7caa7254-5050-3b5e-9eae-bd5ea30e809c,d78e84ec-30aa-3bba-a33a-f29a3a454662,outpatient,2011-01-05
4,5,3de74169-7f67-9304-91d4-757e0f3a14d2,9de5f0b0-4ba4-ce6f-45fb-b55c202f31a5,2011-01-05 12:58:36+00:00,2011-01-05 16:42:36+00:00,265764009,Renal dialysis (procedure),1255,3.733333,2011-01-05,False,b1c428d6-4f07-31e0-90f0-68ffa6ff8c76,d78e84ec-30aa-3bba-a33a-f29a3a454662,ambulatory,2011-01-05


Procedure records: 47701
Unique procedure IDs: 47701
Unmatched encounter references: 0
Missing payer references: 0
Missing encounter classes: 0


## 3. Create the Dimension Tables

Dimension tables provide descriptive attributes and reference keys for filtering and grouping the fact tables.

The model contains five dimensions: patients, payers, organisations, encounter classes and dates.

Patient-identifying information is excluded from the analytical model because it is not required for the main analysis.

In [6]:
# 1. Patient dimension
dim_patients = (
    patients[["Id"]]
    .drop_duplicates()
    .rename(columns={"Id": "PATIENT"})
    .reset_index(drop=True)
)

# 2. Payer dimension
dim_payers = (
    payers[["Id", "NAME"]]
    .drop_duplicates()
    .rename(columns={
        "Id": "PAYER",
        "NAME": "PAYER_NAME"
    })
    .reset_index(drop=True)
)

# 3. Organisation dimension
dim_organizations = (
    organizations[["Id", "NAME", "CITY", "STATE"]]
    .drop_duplicates()
    .rename(columns={
        "Id": "ORGANIZATION",
        "NAME": "ORGANIZATION_NAME"
    })
    .reset_index(drop=True)
)

# 4. Encounter class dimension
dim_encounter_class = (
    fact_encounters[["ENCOUNTERCLASS"]]
    .drop_duplicates()
    .sort_values("ENCOUNTERCLASS")
    .reset_index(drop=True)
)

# Display dimension sizes
dimension_sizes = pd.DataFrame({
    "Dimension": [
        "DimPatients",
        "DimPayers",
        "DimOrganizations",
        "DimEncounterClass"
    ],
    "Rows": [
        len(dim_patients),
        len(dim_payers),
        len(dim_organizations),
        len(dim_encounter_class)
    ]
})

display(dimension_sizes)

,Dimension,Rows
0,DimPatients,974
1,DimPayers,10
2,DimOrganizations,1
3,DimEncounterClass,6


In [7]:
# Determine the full date range across both fact tables
earliest_date = min(
    fact_encounters["START_DATE"].min(),
    fact_procedures["PROCEDURE_DATE"].min()
)

latest_date = max(
    fact_encounters["START_DATE"].max(),
    fact_procedures["PROCEDURE_DATE"].max()
)

# Generate a continuous calendar
date_range = pd.date_range(
    start=earliest_date,
    end=latest_date,
    freq="D"
)

# Create the date dimension
dim_date = pd.DataFrame({
    "DATE": date_range
})

dim_date["YEAR"] = dim_date["DATE"].dt.year
dim_date["QUARTER"] = (
    "Q" + dim_date["DATE"].dt.quarter.astype(str)
)
dim_date["MONTH_NUMBER"] = dim_date["DATE"].dt.month
dim_date["MONTH_NAME"] = dim_date["DATE"].dt.month_name()
dim_date["YEAR_MONTH"] = dim_date["DATE"].dt.strftime("%Y-%m")
dim_date["DAY"] = dim_date["DATE"].dt.day
dim_date["DAY_NAME"] = dim_date["DATE"].dt.day_name()

display(dim_date.head())
display(dim_date.tail())

print("Earliest date:", dim_date["DATE"].min())
print("Latest date:", dim_date["DATE"].max())
print("Total calendar dates:", len(dim_date))

,DATE,YEAR,QUARTER,MONTH_NUMBER,MONTH_NAME,YEAR_MONTH,DAY,DAY_NAME
0,2011-01-02,2011,Q1,1,January,2011-01,2,Sunday
1,2011-01-03,2011,Q1,1,January,2011-01,3,Monday
2,2011-01-04,2011,Q1,1,January,2011-01,4,Tuesday
3,2011-01-05,2011,Q1,1,January,2011-01,5,Wednesday
4,2011-01-06,2011,Q1,1,January,2011-01,6,Thursday


,DATE,YEAR,QUARTER,MONTH_NUMBER,MONTH_NAME,YEAR_MONTH,DAY,DAY_NAME
4048,2022-02-01,2022,Q1,2,February,2022-02,1,Tuesday
4049,2022-02-02,2022,Q1,2,February,2022-02,2,Wednesday
4050,2022-02-03,2022,Q1,2,February,2022-02,3,Thursday
4051,2022-02-04,2022,Q1,2,February,2022-02,4,Friday
4052,2022-02-05,2022,Q1,2,February,2022-02,5,Saturday


Earliest date: 2011-01-02 00:00:00
Latest date: 2022-02-05 00:00:00
Total calendar dates: 4053


In [8]:
dimension_checks = {
    "DimPatients": {
        "Rows": len(dim_patients),
        "Unique keys": dim_patients["PATIENT"].nunique(),
        "Missing keys": dim_patients["PATIENT"].isna().sum()
    },
    "DimPayers": {
        "Rows": len(dim_payers),
        "Unique keys": dim_payers["PAYER"].nunique(),
        "Missing keys": dim_payers["PAYER"].isna().sum()
    },
    "DimOrganizations": {
        "Rows": len(dim_organizations),
        "Unique keys": dim_organizations["ORGANIZATION"].nunique(),
        "Missing keys": dim_organizations["ORGANIZATION"].isna().sum()
    },
    "DimEncounterClass": {
        "Rows": len(dim_encounter_class),
        "Unique keys": dim_encounter_class["ENCOUNTERCLASS"].nunique(),
        "Missing keys": dim_encounter_class["ENCOUNTERCLASS"].isna().sum()
    },
    "DimDate": {
        "Rows": len(dim_date),
        "Unique keys": dim_date["DATE"].nunique(),
        "Missing keys": dim_date["DATE"].isna().sum()
    }
}

dimension_validation = pd.DataFrame(dimension_checks).T

display(dimension_validation)

,Rows,Unique keys,Missing keys
DimPatients,974,974,0
DimPayers,10,10,0
DimOrganizations,1,1,0
DimEncounterClass,6,6,0
DimDate,4053,4053,0


## 4. Final Model Validation

The final validation checks confirm that the fact tables retain their original record counts, dimension keys are unique and all foreign key relationships are valid.

The procedure-to-encounter relationship and date coverage are also checked before the analytical tables are exported.

In [9]:
# Define all dimension relationships
foreign_key_checks = [
    ("FactEncounters", fact_encounters, "PATIENT",
     "DimPatients", dim_patients, "PATIENT"),

    ("FactEncounters", fact_encounters, "PAYER",
     "DimPayers", dim_payers, "PAYER"),

    ("FactEncounters", fact_encounters, "ORGANIZATION",
     "DimOrganizations", dim_organizations, "ORGANIZATION"),

    ("FactEncounters", fact_encounters, "ENCOUNTERCLASS",
     "DimEncounterClass", dim_encounter_class, "ENCOUNTERCLASS"),

    ("FactProcedures", fact_procedures, "PATIENT",
     "DimPatients", dim_patients, "PATIENT"),

    ("FactProcedures", fact_procedures, "PAYER",
     "DimPayers", dim_payers, "PAYER"),

    ("FactProcedures", fact_procedures, "ORGANIZATION",
     "DimOrganizations", dim_organizations, "ORGANIZATION"),

    ("FactProcedures", fact_procedures, "ENCOUNTERCLASS",
     "DimEncounterClass", dim_encounter_class, "ENCOUNTERCLASS")
]

# Run relationship validation
relationship_results = []

for fact_name, fact_df, fact_key, dim_name, dim_df, dim_key in foreign_key_checks:

    unmatched = ~fact_df[fact_key].isin(dim_df[dim_key])

    relationship_results.append({
        "Fact Table": fact_name,
        "Foreign Key": fact_key,
        "Dimension": dim_name,
        "Unmatched Records": unmatched.sum(),
        "Status": "PASS" if unmatched.sum() == 0 else "CHECK"
    })

relationship_validation = pd.DataFrame(relationship_results)

display(relationship_validation)

,Fact Table,Foreign Key,Dimension,Unmatched Records,Status
0,FactEncounters,PATIENT,DimPatients,0,PASS
1,FactEncounters,PAYER,DimPayers,0,PASS
2,FactEncounters,ORGANIZATION,DimOrganizations,0,PASS
3,FactEncounters,ENCOUNTERCLASS,DimEncounterClass,0,PASS
4,FactProcedures,PATIENT,DimPatients,0,PASS
5,FactProcedures,PAYER,DimPayers,0,PASS
6,FactProcedures,ORGANIZATION,DimOrganizations,0,PASS
7,FactProcedures,ENCOUNTERCLASS,DimEncounterClass,0,PASS


In [10]:
# Check procedure encounter references
unmatched_encounters = ~fact_procedures["ENCOUNTER"].isin(
    fact_encounters["Id"]
)

# Check patient consistency
encounter_patient_lookup = fact_encounters.set_index("Id")["PATIENT"]

expected_patients = fact_procedures["ENCOUNTER"].map(
    encounter_patient_lookup
)

patient_mismatches = (
    fact_procedures["PATIENT"] != expected_patients
)

print("Unmatched encounter references:", unmatched_encounters.sum())
print("Patient reference mismatches:", patient_mismatches.sum())

Unmatched encounter references: 0
Patient reference mismatches: 0


In [11]:
calendar_dates = set(dim_date["DATE"].dt.date)

encounter_dates = set(fact_encounters["START_DATE"])
procedure_dates = set(fact_procedures["PROCEDURE_DATE"])

print(
    "Encounter dates outside calendar:",
    len(encounter_dates - calendar_dates)
)

print(
    "Procedure dates outside calendar:",
    len(procedure_dates - calendar_dates)
)

Encounter dates outside calendar: 0
Procedure dates outside calendar: 0


In [12]:
final_model_checks = pd.DataFrame([
    {
        "Table": "FactEncounters",
        "Expected Rows": len(encounters),
        "Actual Rows": len(fact_encounters),
        "Unique Primary Keys": fact_encounters["Id"].nunique(),
        "Missing Primary Keys": fact_encounters["Id"].isna().sum()
    },
    {
        "Table": "FactProcedures",
        "Expected Rows": len(procedures),
        "Actual Rows": len(fact_procedures),
        "Unique Primary Keys": fact_procedures["PROCEDURE_ID"].nunique(),
        "Missing Primary Keys": fact_procedures["PROCEDURE_ID"].isna().sum()
    }
])

display(final_model_checks)

,Table,Expected Rows,Actual Rows,Unique Primary Keys,Missing Primary Keys
0,FactEncounters,27891,27891,27891,0
1,FactProcedures,47701,47701,47701,0


## 14. Export Validated Analytical Tables

The validated fact and dimension tables are exported as CSV files for use in Power BI. Each table is saved separately to preserve the analytical model structure.

In [13]:
model_tables = {
    "fact_encounters": fact_encounters,
    "fact_procedures": fact_procedures,
    "dim_patients": dim_patients,
    "dim_payers": dim_payers,
    "dim_organizations": dim_organizations,
    "dim_encounter_class": dim_encounter_class,
    "dim_date": dim_date
}

for table_name, dataframe in model_tables.items():
    output_path = PROCESSED_DATA_DIR / f"{table_name}.csv"
    dataframe.to_csv(output_path, index=False)

    print(
        f"{table_name}.csv | "
        f"{len(dataframe):,} rows | "
        f"{len(dataframe.columns)} columns | Saved"
    )

fact_encounters.csv | 27,891 rows | 14 columns | Saved
fact_procedures.csv | 47,701 rows | 15 columns | Saved
dim_patients.csv | 974 rows | 1 columns | Saved
dim_payers.csv | 10 rows | 2 columns | Saved
dim_organizations.csv | 1 rows | 4 columns | Saved
dim_encounter_class.csv | 6 rows | 1 columns | Saved
dim_date.csv | 4,053 rows | 8 columns | Saved


In [14]:
exported_files = sorted(PROCESSED_DATA_DIR.glob("*.csv"))

print("Number of exported CSV files:", len(exported_files))

for file in exported_files:
    print(file.name, "|", f"{file.stat().st_size / 1024:.2f} KB")

Number of exported CSV files: 7
dim_date.csv | 191.06 KB
dim_encounter_class.csv | 0.08 KB
dim_organizations.csv | 0.12 KB
dim_patients.csv | 36.15 KB
dim_payers.csv | 0.50 KB
fact_encounters.csv | 7239.74 KB
fact_procedures.csv | 14695.47 KB
